# Bank Churn: Analysis

**Business question:** which customers are about to leave the bank, and how should the retention team spend its limited time?

This notebook explores the data, then reads the results produced by `train.py`.
Run `python train.py` once before section 3.

In [ ]:
import json
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import ConfusionMatrixDisplay, precision_recall_curve
from sklearn.model_selection import train_test_split

import train  # reuse the same feature lists, seed and data loader as training

df = pd.read_csv("data/Churn_Modelling.csv")
df.head()

## 1. How many customers leave?

In [ ]:
churn_rate = df["Exited"].mean()
print(f"{len(df):,} customers, churn rate = {churn_rate:.1%}")
print(f"A model that always says 'nobody leaves' is already {1 - churn_rate:.1%} accurate, so accuracy alone is misleading.")

## 2. Who leaves?

In [ ]:
df["AgeBand"] = pd.cut(df["Age"], bins=[17, 30, 40, 50, 60, 100], labels=["18-30", "31-40", "41-50", "51-60", "60+"])

fig, axes = plt.subplots(2, 3, figsize=(14, 7))
for ax, col in zip(axes.ravel(), ["Geography", "Gender", "AgeBand", "IsActiveMember", "NumOfProducts", "HasCrCard"]):
    rates = df.groupby(col, observed=True)["Exited"].mean()
    rates.plot.bar(ax=ax, color="#8a6a1f")
    ax.axhline(churn_rate, color="gray", linestyle="--", linewidth=1)
    ax.set_title(f"Churn rate by {col}")
    ax.set_ylabel("churn rate")
    ax.tick_params(axis="x", rotation=0)
plt.tight_layout()
plt.show()

> ✍️ **Your observations (write 3, in your own words):**
> 1. Which country has the highest churn rate, and by how much compared with the average?
> 2. What happens to churn as customers get older?
> 3. What does `IsActiveMember` tell the retention team?

## 3. Model results (from `train.py`)

In [ ]:
metrics = json.load(open("reports/metrics.json"))
rows = ["baseline_nobody_leaves", "ann_threshold_0.5", "ann_tuned", "xgboost_tuned"]
cols = ["threshold", "accuracy", "precision", "recall", "f1", "roc_auc", "pr_auc", "churners_caught_in_top_20pct"]
pd.DataFrame({r: metrics[r] for r in rows}).T[cols]

**How to read this table**
- **Recall**: of the customers who really left, how many did we flag?
- **Precision**: of the customers we flagged, how many really left? (the rest are wasted retention offers)
- **Threshold**: the probability above which we call a customer "high risk". It was tuned on the *validation* set to reach at least 70% recall; the test set was only used once, for these final numbers.

## 4. Precision vs. recall: choosing the threshold

In [ ]:
# Rebuild the exact same test split as train.py (same seed, same stratification).
X, y = train.load_data()
_, X_test, _, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=train.SEED)

pre = joblib.load("models/preprocessor.joblib")
model = tf.keras.models.load_model("models/churn_ann.keras")
threshold = json.load(open("models/threshold.json"))["threshold"]
proba = model.predict(pre.transform(X_test), verbose=0).ravel()

precision, recall, thresholds = precision_recall_curve(y_test, proba)
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
ax[0].plot(thresholds, precision[:-1], label="precision")
ax[0].plot(thresholds, recall[:-1], label="recall")
ax[0].axvline(threshold, color="gray", linestyle="--", label=f"chosen threshold = {threshold:.2f}")
ax[0].set_xlabel("threshold")
ax[0].set_title("Moving the threshold trades precision for recall")
ax[0].legend()

ConfusionMatrixDisplay.from_predictions(
    y_test, (proba >= threshold).astype(int), display_labels=["stays", "leaves"], ax=ax[1], colorbar=False
)
ax[1].set_title("ANN on the test set (tuned threshold)")
plt.tight_layout()
plt.show()

## 5. Business view: who should the retention team call first?

In [ ]:
order = np.argsort(proba)[::-1]
caught = np.cumsum(y_test.to_numpy()[order]) / y_test.sum()
contacted = np.arange(1, len(order) + 1) / len(order)

plt.figure(figsize=(6.5, 4.5))
plt.plot(contacted, caught, label="ANN ranking")
plt.plot([0, 1], [0, 1], "--", color="gray", label="random calls")
plt.axvline(0.2, color="#8a6a1f", linestyle=":")
plt.xlabel("share of customers contacted (riskiest first)")
plt.ylabel("share of leavers reached")
plt.title("Calling the riskiest customers first")
plt.legend()
plt.show()

top20 = caught[int(0.2 * len(order)) - 1]
print(f"Calling the riskiest 20% of customers reaches {top20:.0%} of everyone who would leave (random calls: 20%).")

> ✍️ **Business recommendation (your words, 3–4 lines):**
> If the retention team can only call 20% of customers, what should they do, and what result can they expect?
> This paragraph goes into the README and becomes the CV bullet.